In [ ]:
#importaciones y Librerias 
import pandas as pd
from sklearn.model_selection import train_test_split
import unicodedata
import re
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
import torch.nn as nn
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

In [4]:

dataset = pd.read_json("dataset/dataset_humor_train.json", lines=True)
#conteo de clases
print("Ejemplos de entrenamiento: ")
print(dataset.klass.value_counts())


Ejemplos de entrenamiento: 
klass
0    6588
1    3812
Name: count, dtype: int64


In [8]:
# Extracción de los textos en arreglos de numpy
X_train = dataset['text'].to_numpy()
# Extracción de las etiquetas o clases de entrenamiento
Y_train = dataset['klass'].to_numpy()

In [9]:
# Divide el conjunto de entrenamiento en:  entrenamiento (90%) y validación (10%)
X_train, X_val, Y_train, Y_val =  train_test_split(X_train, Y_train, test_size=0.1, stratify=Y_train, random_state=42)

In [ ]:
_STOPWORDS = stopwords.words("english")  # agregar más palabras a esta lista si es necesario
stemmer = SnowballStemmer("english")
PUNCTUACTION = ";:,.\\-\"'/"
SYMBOLS = "()[]¿?¡!{}~<>|"
NUMBERS= "0123456789"
SKIP_SYMBOLS = set(PUNCTUACTION + SYMBOLS)
SKIP_SYMBOLS_AND_SPACES = set(PUNCTUACTION + SYMBOLS + '\t\n\r ')

def normaliza_texto(input_str,
                    punct=False,
                    accents=False,
                    num=False,
                    max_dup=2):
    """
        punct=False (elimina la puntuación, True deja intacta la puntuación)
        accents=False (elimina los acentos, True deja intactos los acentos)
        num= False (elimina los números, True deja intactos los acentos)
        max_dup=2 (número máximo de símbolos duplicados de forma consecutiva, rrrrr => rr)
    """
    
    nfkd_f = unicodedata.normalize('NFKD', input_str)
    n_str = []
    c_prev = ''
    cc_prev = 0
    for c in nfkd_f:
        if not num:
            if c in NUMBERS:
                continue
        if not punct:
            if c in SKIP_SYMBOLS:
                continue
        if not accents and unicodedata.combining(c):
            continue
        if c_prev == c:
            cc_prev += 1
            if cc_prev >= max_dup:
                continue
        else:
            cc_prev = 0
        n_str.append(c)
        c_prev = c
    texto = unicodedata.normalize('NFKD', "".join(n_str))
    texto = re.sub(r'(\s)+', r' ', texto.strip(), flags=re.IGNORECASE)
    return texto

def mi_preprocesamiento_factory(modo):
    """
    Devuelve una función de preprocesamiento y tokenización según el modo:
    - 'normalizacion'
    - 'normalizacion_stopwords'
    - 'normalizacion_stopwords_stem'
    """

    def preprocesamiento(texto):
        texto = texto.lower()
        #texto = eliminar_emojis(texto)
        texto = re.sub(r"http\S+|www\S+|https\S+", "", texto)
        texto = re.sub(r"@\w+", "", texto)
        texto = normaliza_texto(texto, punct=True)
        return texto

    def tokenizador(texto):
        tokens = word_tokenize(texto)
        if modo in ['normalizacion_stopwords', 'normalizacion_stopwords_stem']:
            tokens = [t for t in tokens if t not in _STOPWORDS and len(t) > 2]
        if modo == 'normalizacion_stopwords_stem':
            tokens = [stemmer.stem(t) for t in tokens]
        return tokens

    return preprocesamiento, tokenizador

In [ ]:
class Vec_TFID:
    def __init__(self, modo_preproc='normalizacion'):
        self.vec_tfidf = None
        self.modo_preproc = modo_preproc

    def create_matriz_TFID(self, X_train, ngram_config, max_config):
        preproc, token = mi_preprocesamiento_factory(self.modo_preproc)
        self.vec_tfidf = TfidfVectorizer(
            analyzer="word",
            preprocessor=preproc,
            tokenizer=token,
            ngram_range=ngram_config,
            max_features=max_config
        )
        X_tfidf = self.vec_tfidf.fit_transform(X_train)
        return X_tfidf.toarray()

    def tranform_matriz_TFID(self, X_test):
        X_tfid = self.vec_tfidf.transform(X_test)
        return X_tfid.toarray()

In [ ]:
class Vec_Count:
    def __init__(self, modo_preproc='normalizacion'):
        self.vec = None
        self.modo_preproc = modo_preproc

    def create_matriz_Count(self, X_train, ngram_config, max_config):
        preproc, token = mi_preprocesamiento_factory(self.modo_preproc)
        self.vec = CountVectorizer(
            analyzer="word",
            preprocessor=preproc,
            tokenizer=token,
            ngram_range=ngram_config,
            max_features=max_config
        )
        X_vec = self.vec.fit_transform(X_train)
        return X_vec.toarray().astype(np.float32)

    def transform_matriz_Count(self, X_test):
        X_vec = self.vec.transform(X_test)
        return X_vec.toarray().astype(np.float32)


In [ ]:
if USE_TFIDF: 
    vectorizer = Vec_TFID(modo_preproc=preproc)

    X_tr = vectorizer.create_matriz_TFID(X_train, ngram_config=NGRAM_RANGE, 
                                            max_config=MAX_FEATURES)
    
    X_val_vectorized = vectorizer.tranform_matriz_TFID(X_val)
    X_t = vectorizer.tranform_matriz_TFID(X_test) 
else:
    vectorizer = Vec_Count(modo_preproc=preproc)
    X_tr = vectorizer.create_matriz_Count(X_train, 
                                            ngram_config=NGRAM_RANGE, 
                                            max_config=MAX_FEATURES)
    
    X_val_vectorized = vectorizer.transform_matriz_Count(X_val)
    X_t = vectorizer.transform_matriz_Count(X_test)

In [ ]:
def create_minibatches(X, Y, batch_size):
    # Recibe los documentos en X y las etiquetas en Y
    dataset = TensorDataset(X, Y) # Cargar los datos en un dataset de tensores
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    # loader = DataLoader(dataset, batch_size=batch_size)
    return loader

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        # Definición de capas, funciones de activación e inicialización de pesos
        input_size_h1 = 512
        input_size_h2 = 128
        input_size_h3 = 32
        self.fc1 = nn.Linear(input_size, input_size_h1)
        self.bn1 = nn.BatchNorm1d(input_size_h1)
        # PReLU tiene parámetros aprendibles: Se recomienda una función de activación independiente por capa
        self.act1= nn.LeakyReLU()
        self.drop1 = nn.Dropout(p=0.5)

        self.fc2 = nn.Linear(input_size_h1, input_size_h2)
        self.bn2 = nn.BatchNorm1d(input_size_h2)
        # PReLU tiene parámetros aprendibles: Se recomienda una función de activación independiente por capa
        self.act2= nn.LeakyReLU()
        self.drop2 = nn.Dropout(p=0.4)

        self.fc3 = nn.Linear(input_size_h2, input_size_h3)
        self.bn3 = nn.BatchNorm1d(input_size_h3)
        self.act3 = nn.LeakyReLU()
        self.drop3 = nn.Dropout(p=0.2)

        self.output = nn.Linear(input_size_h3, output_size)
        
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
       
    def forward(self, X):
        # Definición del orden de conexión de las capas y aplición de las funciones de activación
        x = self.fc1(X)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.drop1(x)


        x = self.fc2(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.drop2(x)

        
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.act3(x)
        x = self.drop3(x)

        x = self.output(x)
        # Nota la última capa de salida 'output' no se activa debido a que CrossEntropyLoss usa LogSoftmax internamente. 
        return x